> ## SOLUTION / ANSWER KEY &mdash; Lab 8.3
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-03-shared-state.ipynb`](../lab-03-shared-state.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.3 &mdash; Shared State & Message Passing

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Record each agent's finding into a shared state
- Keep an ordered log of who said what
- Expose a small, relevant context for the next agent

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Message passing & shared state](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = "/tmp/biaa-lab-08-03"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Agents coordinate by **communicating** (deck slide 7): they pass **messages** or read/write a **shared
state** &mdash; a common object carrying the conversation, each agent's findings, and the plan. **LangGraph**
works exactly this way: the state flows through every node. Two rules keep it sane: keep the shared context
**small &amp; relevant** (don't dump everything to everyone), and make handoffs **explicit**.

In [ ]:
# One finding is just (agent_name, what it found).
print("a finding:", ("billing", "duplicate charge on 4471"))

## Build it
Complete `SharedState`: record findings (keyed by agent), keep an ordered log, and return a small
context.

In [ ]:
class SharedState:
    def __init__(self, message):
        self.message = message
        self.findings = {}
        self.log = []
    def record(self, agent, finding):
        self.findings[agent] = finding
        self.log.append((agent, finding))
    def context(self):
        # small & relevant: just the findings gathered so far
        return self.findings

try:
    st = SharedState("charged twice and the app crashes")
    st.record("billing", "duplicate charge on 4471")
    st.record("tech", "matches BUG-231")
    print("findings:", st.context())
    print("log     :", st.log)
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Two agents' findings **coexist** in `findings`, keyed by who found them &mdash; nobody overwrites anybody.
- The **`log`** preserves order, so you can replay *who said what, when* &mdash; the seed of observability (Lab 8.10).
- `context()` stays **small** &mdash; only what's been recorded &mdash; so the next agent isn't drowned in irrelevant state.

## Your turn (open task &mdash; no grader)
Add a `summary()` method that returns a one-line string joining every finding (e.g.
`"billing: ...; tech: ..."`), and call it after recording two findings. **What good looks like:** the summary
reads back in a stable order and contains every recorded finding &mdash; the raw material a **synthesiser**
(Lab 8.9) will turn into one reply.

---
### Key takeaway
Shared state is how a team stays coherent -- each agent writes its finding, the next reads the context. Keep it small and let handoffs be explicit, or agents talk past each other.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>